# Claude Agent SDK: streaming output (partial messages)

By default the SDK gives you a **complete** `AssistantMessage` for each content block (a text block, a tool call) **after Claude has finished generating that block**. For a chat UI that feels slow: nothing appears until a whole paragraph is done.

**Partial message streaming** gives you the tokens **as they are generated**.

> Don't confuse this with *streaming input* (how you *send* messages). This notebook is about **output**: how you *receive* them.

**What you will learn (and see running):**
1. Turning streaming on: `include_partial_messages=True`
2. What a `StreamEvent` is, and the six event types you will meet
3. The exact **order** messages arrive in
4. Streaming **text**, then streaming **tool calls** (input JSON arrives in pieces)
5. Building a small **streaming UI** (`[Using Read...] done`)
6. Handling a stream that gets **cut off**, and the other limitations

## The idea in one picture

```
 DEFAULT                                 include_partial_messages=True
 ───────                                 ──────────────────────────────
 ...wait...                              StreamEvent  "I'll"
 AssistantMessage("I'll read the file")  StreamEvent  " read"
                                         StreamEvent  " the file"      ← tokens as they arrive
                                         AssistantMessage("I'll read the file")  ← still delivered
```

You get **extra** `StreamEvent` messages in addition to the usual `AssistantMessage` and `ResultMessage`. Nothing is taken away. And the events are **raw** Claude API events: *you* must accumulate the deltas into text.

## 0. Setup

Run these four cells first. They check Python, install the SDK, load your API key from a `.env` file, and create a private working folder for the demos.

In [ ]:
import sys
print(sys.version)
assert sys.version_info >= (3, 10), "The Agent SDK needs Python 3.10 or newer"

In [ ]:
%pip install -q claude-agent-sdk python-dotenv 

Create a file named `.env` next to this notebook containing one line (no quotes, no spaces around `=`):

```
ANTHROPIC_API_KEY=your-api-key
```

In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv, find_dotenv

env_path = find_dotenv(usecwd=True)
load_dotenv(env_path, override=True)

key = os.environ.get("ANTHROPIC_API_KEY", "")
if not key:
    raise RuntimeError(
        f"ANTHROPIC_API_KEY not found. Create a .env file in {Path.cwd()} with:\n"
        "ANTHROPIC_API_KEY=your-api-key"
    )
print("Loaded .env from:", env_path)
print("API key loaded:", key[:7] + "..." + key[-4:])

In [ ]:
from pathlib import Path

DEMO_DIR = Path("streaming-output-demo").resolve()
DEMO_DIR.mkdir(exist_ok=True)
print("Demo folder:", DEMO_DIR)

In [ ]:
(DEMO_DIR / "README.md").write_text(
    "# Streaming demo\n\nThis folder exists so the agent has something to read.\n"
    "It contains a README and a small notes file.\n"
)
(DEMO_DIR / "notes.txt").write_text("Remember to water the plants.\n")
print(sorted(p.name for p in DEMO_DIR.iterdir()))

In [ ]:
# OFFLINE
import json, time
from claude_agent_sdk import (
    query, ClaudeAgentOptions, StreamEvent,
    AssistantMessage, UserMessage, ResultMessage, TextBlock, ToolUseBlock,
)

def short(value, n=100):
    text = str(value).replace("\n", " ")
    return text if len(text) <= n else text[: n - 1] + "…"

## 1. Enable it

```python
ClaudeAgentOptions(include_partial_messages=True)
```

Then, for each message your loop receives:
1. Is it a `StreamEvent`? (Python: `isinstance(message, StreamEvent)`.)
2. Look at `message.event`, a **dict**, and its `"type"`.
3. For `content_block_delta` events, check `delta["type"]`: `text_delta` carries `delta["text"]`.

Our first stream:

In [ ]:
t0 = time.perf_counter()
t_first_delta = t_first_block = None
chunks = 0

async for message in query(
    prompt="In about 100 words, explain what a context window is.",
    options=ClaudeAgentOptions(include_partial_messages=True, cwd=str(DEMO_DIR)),
):
    if isinstance(message, StreamEvent):
        event = message.event
        if event.get("type") == "content_block_delta":
            delta = event.get("delta", {})
            if delta.get("type") == "text_delta":
                if t_first_delta is None:
                    t_first_delta = time.perf_counter() - t0
                chunks += 1
                print(delta.get("text", ""), end="", flush=True)
    elif isinstance(message, AssistantMessage) and t_first_block is None:
        t_first_block = time.perf_counter() - t0

total = time.perf_counter() - t0
print(f"\n\nfirst text chunk arrived : {t_first_delta:.2f}s   ({chunks} chunks)")
print(f"first complete block     : {t_first_block:.2f}s   (what you would wait for WITHOUT streaming)")
print(f"whole run                : {total:.2f}s")

The first chunk arrives **before, or at the same moment as,** the first complete block, and the gap grows with the length of the reply. Streaming matters most for long answers and slow tool steps. (If both numbers look nearly equal on your machine, try a longer prompt.) The point is that the user can start reading immediately.

## 2. What is inside a `StreamEvent`?

A `StreamEvent` has `uuid`, `session_id`, `event` (the raw API event) and `parent_tool_use_id`.

| `event["type"]` | Meaning |
|---|---|
| `message_start` | A new message begins |
| `content_block_start` | A new block begins: text **or** `tool_use` (the block carries the tool name) |
| `content_block_delta` | An **incremental update** to the current block |
| `content_block_stop` | That block is finished |
| `message_delta` | Message-level update (stop reason, usage) |
| `message_stop` | The message is finished |

And the kinds of `delta` you will see inside `content_block_delta`:

| `delta["type"]` | Payload |
|---|---|
| `text_delta` | `delta["text"]`: a few tokens of prose |
| `input_json_delta` | `delta["partial_json"]`: a **fragment** of a tool call's JSON input |

Notes:
- `parent_tool_use_id` is **always `None`** on stream events, and stream events are emitted for the **main session only**: token deltas from **subagents are not forwarded**. To attribute output to a subagent, use complete messages (they carry `parent_tool_use_id`).
- The events are **raw**: `partial_json` fragments are not valid JSON until the block stops.

## 3. The order messages arrive in

```text
StreamEvent (message_start)
StreamEvent (content_block_start)  - text block
StreamEvent (content_block_delta)  - text chunks...
AssistantMessage                   - complete text block
StreamEvent (content_block_stop)
StreamEvent (content_block_start)  - tool_use block
StreamEvent (content_block_delta)  - tool input chunks...
AssistantMessage                   - complete tool_use block
StreamEvent (content_block_stop)
StreamEvent (message_delta)
StreamEvent (message_stop)
... the tool executes ...
... more streaming events for the next turn ...
ResultMessage
```

Each `AssistantMessage` carries **one block**, arrives **before** that block's `content_block_stop`, and messages from one response share a **message ID** (`message.message_id` in Python).

Let's verify this on a real, tool-using run. The cell records everything and prints a compressed timeline (`x3` means three in a row).

In [ ]:
def label(message):
    if isinstance(message, StreamEvent):
        e = message.event
        t = e.get("type")
        if t == "content_block_start":
            return f"block_start({e['content_block']['type']})"
        if t == "content_block_delta":
            return f"delta({e['delta']['type']})"
        return t
    if isinstance(message, AssistantMessage):
        kinds = "+".join(type(b).__name__ for b in message.content)
        return f"AssistantMessage[{kinds}]"
    return type(message).__name__


timeline, parents, message_ids = [], set(), set()
async for message in query(
    prompt="Read README.md and tell me, in one sentence, what this folder is for.",
    options=ClaudeAgentOptions(include_partial_messages=True, allowed_tools=["Read"], cwd=str(DEMO_DIR)),
):
    timeline.append(label(message))
    if isinstance(message, StreamEvent):
        parents.add(message.parent_tool_use_id)
    if isinstance(message, AssistantMessage):
        message_ids.add(message.message_id)

# run-length compress
compressed, prev, count = [], None, 0
for item in timeline + [None]:
    if item == prev:
        count += 1
    else:
        if prev is not None:
            compressed.append(prev + (f" x{count}" if count > 1 else ""))
        prev, count = item, 1

for line in compressed:
    print(" ", line)
print("\nparent_tool_use_id values on stream events:", parents)
print("distinct assistant message ids:", len(message_ids))

Check the pattern against the diagram: deltas, then an `AssistantMessage`, then `content_block_stop`; tool block next; a `UserMessage` carrying the tool result; then a second round of streaming for the final answer.

## 4. Accumulating blocks: a reusable assembler

Because events are raw, you need a small state machine that **keeps one buffer per block index** and finalizes it on `content_block_stop`. This also handles the tricky part: tool input arrives as JSON **fragments**.

It lives in plain Python, so we can test it **offline** with a simulated stream.

In [ ]:
# OFFLINE
class BlockAssembler:
    """Turn raw stream events into finished text blocks and tool calls."""

    def __init__(self):
        self.open = {}          # index -> buffer for blocks still streaming
        self.done = []          # finished blocks
        self.incomplete = []    # blocks that were still open at message_stop

    def feed(self, event):
        kind, idx = event.get("type"), event.get("index")

        if kind == "content_block_start":
            block = event["content_block"]
            self.open[idx] = {"type": block["type"], "name": block.get("name"), "text": "", "json": ""}

        elif kind == "content_block_delta":
            buf, delta = self.open.get(idx), event["delta"]
            if buf is None:
                return
            if delta["type"] == "text_delta":
                buf["text"] += delta["text"]
            elif delta["type"] == "input_json_delta":
                buf["json"] += delta["partial_json"]

        elif kind == "content_block_stop":
            buf = self.open.pop(idx, None)
            if buf is not None:
                if buf["type"] == "tool_use":
                    buf["input"] = json.loads(buf["json"] or "{}")
                self.done.append(buf)

        elif kind == "message_stop":
            for buf in self.open.values():      # still open here = the stream was cut off
                buf["incomplete"] = True
                self.incomplete.append(buf)
            self.open.clear()


def fake_stream(truncate_tool=False):
    """Yield StreamEvent objects shaped like a real response: a text block, then a Read call."""
    def ev(e):
        return StreamEvent(uuid="u", session_id="s", event=e, parent_tool_use_id=None)

    yield ev({"type": "message_start", "message": {"id": "m1"}})
    yield ev({"type": "content_block_start", "index": 0, "content_block": {"type": "text", "text": ""}})
    for piece in ["I'll ", "read ", "the file."]:
        yield ev({"type": "content_block_delta", "index": 0, "delta": {"type": "text_delta", "text": piece}})
    yield ev({"type": "content_block_stop", "index": 0})

    yield ev({"type": "content_block_start", "index": 1,
              "content_block": {"type": "tool_use", "id": "t1", "name": "Read", "input": {}}})
    for piece in ['{"file_', 'path": "RE', 'ADME.md"}']:
        yield ev({"type": "content_block_delta", "index": 1,
                  "delta": {"type": "input_json_delta", "partial_json": piece}})
    if not truncate_tool:
        yield ev({"type": "content_block_stop", "index": 1})
    yield ev({"type": "message_delta", "delta": {"stop_reason": "tool_use"}})
    yield ev({"type": "message_stop"})


print("fake_stream defined")

In [ ]:
# OFFLINE
asm = BlockAssembler()
for se in fake_stream():
    asm.feed(se.event)

print("finished blocks:")
for b in asm.done:
    print("  ", b["type"], "->", b["text"] if b["type"] == "text" else (b["name"], b["input"]))
print("incomplete     :", asm.incomplete)

Notice the tool's JSON arrived as `{"file_`, `path": "RE`, `ADME.md"}`: **meaningless until joined**. Parse only after `content_block_stop`.

## 5. When the stream is cut off

If a stream is cut mid-message (you interrupt, the connection drops) you **still receive `message_stop`**. A cut *text* or *thinking* block also gets its `content_block_stop`, but a cut **tool call does not**. So the rule is:

> If `message_stop` arrives while a tool call's block is **still open**, treat that call's input as **incomplete**.

Our assembler does exactly that. Simulate it:

In [ ]:
# OFFLINE
asm = BlockAssembler()
for se in fake_stream(truncate_tool=True):
    asm.feed(se.event)

print("finished:  ", [b["type"] for b in asm.done])
print("incomplete:", [(b["type"], b["name"], b["json"]) for b in asm.incomplete])

(Before Claude Code v2.1.290 a cut stream could end *without* `message_stop`, leaving a reply shown "in progress" forever. The Python SDK bundles v2.1.290+ from v0.2.164. If you see that symptom, **update the SDK**.)

## 6. Streaming tool calls, live

The three events that matter for a tool call:
1. `content_block_start` with `content_block.type == "tool_use"` → the **tool name**
2. `content_block_delta` with `input_json_delta` → **input fragments**
3. `content_block_stop` → the call is **complete**

Here is the live version, using the assembler we just tested.

In [ ]:
asm = BlockAssembler()
async for message in query(
    prompt="Read notes.txt and repeat what it says.",
    options=ClaudeAgentOptions(include_partial_messages=True, allowed_tools=["Read"], cwd=str(DEMO_DIR)),
):
    if isinstance(message, StreamEvent):
        e = message.event
        if e.get("type") == "content_block_start" and e["content_block"]["type"] == "tool_use":
            print(f"Starting tool: {e['content_block']['name']}")
        if e.get("type") == "content_block_delta" and e["delta"]["type"] == "input_json_delta":
            print(f"   input fragment: {e['delta']['partial_json']!r}")
        before = len(asm.done)
        asm.feed(e)
        if len(asm.done) > before and asm.done[-1]["type"] == "tool_use":
            b = asm.done[-1]
            print(f"Tool {b['name']} called with: {b['input']}")

## 7. A streaming UI

Combine text and tool streaming into a chat-style display: stream prose as it arrives, show `[Using Read...]` while a tool call is being generated, then ` done`.

A flag, `in_tool`, flips on `content_block_start(tool_use)` and off on `content_block_stop`. Here is the rendering logic as a pure function so we can **test it offline** against our fake stream:

In [ ]:
# OFFLINE
class StreamingRenderer:
    """Turns stream events into the text a terminal chat UI would print."""

    def __init__(self):
        self.in_tool = False
        self.out = []

    def feed(self, event):
        kind = event.get("type")
        if kind == "content_block_start" and event["content_block"]["type"] == "tool_use":
            self.out.append(f"\n[Using {event['content_block']['name']}...]")
            self.in_tool = True
        elif kind == "content_block_delta":
            delta = event.get("delta", {})
            if delta.get("type") == "text_delta" and not self.in_tool:
                self.out.append(delta["text"])
        elif kind == "content_block_stop" and self.in_tool:
            self.out.append(" done")
            self.in_tool = False

    def text(self):
        return "".join(self.out)


r = StreamingRenderer()
for se in fake_stream():
    r.feed(se.event)
print(r.text())

In [ ]:
import sys

class LivePrinter(StreamingRenderer):
    def feed(self, event):
        n = len(self.out)
        super().feed(event)
        for piece in self.out[n:]:
            sys.stdout.write(piece)
            sys.stdout.flush()

ui = LivePrinter()
async for message in query(
    prompt="Read README.md, then summarize it in one sentence.",
    options=ClaudeAgentOptions(include_partial_messages=True, allowed_tools=["Read"], cwd=str(DEMO_DIR)),
):
    if isinstance(message, StreamEvent):
        ui.feed(message.event)
    elif isinstance(message, ResultMessage):
        print("\n\n--- Complete ---")

## 8. Known limitations and gotchas

- **Structured output.** With partial messages on, the JSON streams as a tool call's **unvalidated** `input_json_delta` chunks. Only the **validated** result reaches `ResultMessage.structured_output`. Never trust streamed fragments as final data (see the next notebook).
- **Subagents.** Their token deltas are **not forwarded**; use complete messages and `parent_tool_use_id`.
- **You must accumulate** deltas yourself; events are raw, not running totals.
- **Extra volume.** Partial messages add many small messages. Skip them for batch jobs and CI where nobody is watching.
- Without partial messages you still get `SystemMessage`, `AssistantMessage`, `ResultMessage` and compaction boundaries, just no `StreamEvent`.

## 9. Recap

| Want | Do |
|---|---|
| Tokens as they arrive | `include_partial_messages=True`, handle `StreamEvent` |
| Text chunks | `content_block_delta` + `text_delta` → `delta["text"]` |
| A tool call's input | accumulate `input_json_delta` → parse at `content_block_stop` |
| Spot a cut-off tool call | `message_stop` while its block is still open |
| Attribute output to a subagent | complete messages' `parent_tool_use_id`, not stream events |

### Try it yourself
1. Show a spinner while `in_tool` is true, and print elapsed time per tool call.
2. Measure time-to-first-chunk for `effort="low"` vs `effort="high"`.
3. Track `message_delta` events and print the final `stop_reason` and usage.
4. Next: **`agent-sdk-structured-output-explained.ipynb`**.